In [1]:
# Install the PyCUDA library to interface Python with the GPU hardware
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.4 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=ca94d134497820602ed196e88f4f7d989b7131e9c086423c6e8bc46b1af186d4
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [2]:
import numpy as np
import time
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# Raw C++ CUDA Kernel Code executed on individual GPU threads
cuda_code = """
#include <curand_kernel.h>

extern "C" {
// 1. Kernel to initialize a unique random number generator state for each thread
__global__ void setup_rand(curandState *state, unsigned long seed) {
    int id = threadIdx.x + blockIdx.x * blockDim.x;
    curand_init(seed, id, 0, &state[id]);
}

// 2. Kernel to simulate asset paths and compute option payoffs
__global__ void monte_carlo_option(curandState *state, float S0, float K, float r,
                                   float sigma, float T, float *d_payoffs, int num_paths) {
    int id = threadIdx.x + blockIdx.x * blockDim.x;
    if (id >= num_paths) return; // Guard clause to prevent out-of-bounds memory access

    // Load the thread's unique random state into local registers
    curandState localState = state[id];
    float rand_norm = curand_normal(&localState);

    // Mathematical Formula: Geometric Brownian Motion (GBM) terminal price
    float ST = S0 * expf((r - 0.5f * sigma * sigma) * T + sigma * sqrtf(T) * rand_norm);

    // Financial Logic: Call Option Payoff = max(ST - K, 0)
    float payoff = ST - K;
    d_payoffs[id] = (payoff > 0.0f) ? payoff : 0.0f;

    // Save the updated random state back to global memory
    state[id] = localState;
}
}
"""

# Compile the C++ code to a live GPU module via the NVIDIA compiler driver (NVCC)
mod = SourceModule(cuda_code, no_extern_c=True)
setup_rand = mod.get_function("setup_rand")
monte_carlo_option = mod.get_function("monte_carlo_option")
print("CUDA kernels compiled successfully!")

CUDA kernels compiled successfully!


In [3]:
# 1. Define Financial and Operational Parameters
num_paths = 10_000_000
S0 = 100.0   # Current Stock Price
K = 105.0    # Strike Price
r = 0.05     # Risk-free interest rate (5%)
sigma = 0.20 # Market Volatility (20%)
T = 1.0      # Time to maturity (1 year)

# 2. Hardware Mapping Configuration
block_size = 256  # Number of threads per block
grid_size = (num_paths + block_size - 1) // block_size  # Total blocks needed

# 3. Allocate Dedicated Memory Blocks on the GPU (Device)
# Each curandStateXORWOW structure takes up 48 bytes of memory
states = cuda.mem_alloc(num_paths * 48)
# 10 million float outputs require 4 bytes per path
d_payoffs = cuda.mem_alloc(num_paths * 4)

# Allocate an empty array on the CPU (Host) to receive the final results
h_payoffs = np.zeros(num_paths, dtype=np.float32)
print(f"Allocated memory arrays for {num_paths:,} concurrent GPU execution pathways.")

Allocated memory arrays for 10,000,000 concurrent GPU execution pathways.


In [4]:
# 1. Execute on the GPU Hardware
start_gpu = time.time()

# Step A: Initialize random number generators in parallel across threads
setup_rand(states, np.uint64(1234), block=(block_size, 1, 1), grid=(grid_size, 1))

# Step B: Compute the path trajectories and option payoffs
monte_carlo_option(states, np.float32(S0), np.float32(K), np.float32(r),
                   np.float32(sigma), np.float32(T), d_payoffs, np.int32(num_paths),
                   block=(block_size, 1, 1), grid=(grid_size, 1))

# Step C: Copy the resulting array from GPU memory back to CPU memory
cuda.memcpy_dtoh(h_payoffs, d_payoffs)

# Step D: Average the results and apply the continuous time-value discount factor e^(-rT)
gpu_price = np.mean(h_payoffs) * np.exp(-r * T)
gpu_time = time.time() - start_gpu

# 2. Run a Mock CPU Baseline to Calculate Speedup Factor
start_cpu = time.time()
cpu_paths = 100_000  # We scale down to 100k paths so the CPU doesn't freeze your tab
Z = np.random.standard_normal(cpu_paths)
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
cpu_price = np.mean(np.maximum(ST - K, 0)) * np.exp(-r * T)
# Scale the time up linearly to approximate how long 10M paths would take on a single CPU core
cpu_time = (time.time() - start_cpu) * 100

# 3. Output Performance Analytics
print(f" \n=== HARDWARE BENCHMARK PROFILE ===")
print(f"Calculated Option Price: ${gpu_price:.4f}")
print(f"GPU Execution Time (10M Paths): {gpu_time:.4f} seconds")
print(f"Estimated CPU Execution Time: {cpu_time:.4f} seconds")
print(f"Parallel Speedup Factor: {cpu_time / gpu_time:.1f}x Faster on GPU Hardware")

 
=== HARDWARE BENCHMARK PROFILE ===
Calculated Option Price: $8.0171
GPU Execution Time (10M Paths): 0.3674 seconds
Estimated CPU Execution Time: 1.1317 seconds
Parallel Speedup Factor: 3.1x Faster on GPU Hardware
